# ONNX: save the model as one file

ONNX is a file format for models. We train in PyTorch, then export to a `.onnx` file.
ONNX Runtime can run that file without PyTorch. It is small and fast.

```
PyTorch model  ->  model.onnx  ->  ONNX Runtime  ->  prediction
```

In [1]:
import os
import time
import warnings
import numpy as np
import torch
import torch.nn as nn
import onnxruntime as ort
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2

LABELS = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
os.makedirs("outputs", exist_ok=True)
warnings.filterwarnings("ignore")

## 1. Load the trained PyTorch model

In [2]:
model = mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(1280, 10)
model.load_state_dict(torch.load("../models/mobilenetv2_cifar10_fp32.pth", map_location="cpu", weights_only=True))
model.eval()
print("model loaded")

model loaded


## 2. Load 200 test pictures

In [3]:
prep = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
test_set = datasets.CIFAR10("../data", train=False, download=True, transform=prep)

images = torch.stack([test_set[i][0] for i in range(200)])
labels = np.array([test_set[i][1] for i in range(200)])
print("images:", images.shape)

images: torch.Size([200, 3, 96, 96])


## 3. Export to ONNX

`torch.onnx.export` runs the model once on a dummy picture and saves all the steps into a file.
`dynamic_axes` means the number of pictures per call can change.

In [4]:
dummy = torch.randn(1, 3, 96, 96)

torch.onnx.export(
    model, dummy, "outputs/model.onnx",
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    dynamo=False,
)

print("file size:", round(os.path.getsize("outputs/model.onnx") / 1e6, 1), "MB")

file size: 8.9 MB


## 4. Run it with ONNX Runtime

Same pictures in PyTorch and in ONNX Runtime. The answers should be the same.

In [5]:
session = ort.InferenceSession("outputs/model.onnx")

x = images[:5].numpy()
onnx_out = session.run(None, {"input": x})[0]

with torch.no_grad():
    torch_out = model(images[:5]).numpy()

print("true   :", [LABELS[i] for i in labels[:5]])
print("PyTorch:", [LABELS[i] for i in torch_out.argmax(1)])
print("ONNX   :", [LABELS[i] for i in onnx_out.argmax(1)])
print("biggest difference in scores:", np.abs(torch_out - onnx_out).max())

true   : ['cat', 'ship', 'ship', 'airplane', 'frog']
PyTorch: ['cat', 'ship', 'ship', 'airplane', 'frog']
ONNX   : ['cat', 'ship', 'ship', 'airplane', 'frog']
biggest difference in scores: 2.3841858e-06


## 5. Accuracy on 200 pictures

In [6]:
with torch.no_grad():
    torch_pred = model(images).argmax(1).numpy()
onnx_pred = session.run(None, {"input": images.numpy()})[0].argmax(1)

print("PyTorch accuracy:", (torch_pred == labels).mean())
print("ONNX accuracy   :", (onnx_pred == labels).mean())

PyTorch accuracy: 0.905
ONNX accuracy   : 0.905


## 6. Speed: one picture, 100 times

In [7]:
one = images[:1]

start = time.time()
with torch.no_grad():
    for _ in range(100):
        model(one)
torch_ms = (time.time() - start) * 1000 / 100

start = time.time()
for _ in range(100):
    session.run(None, {"input": one.numpy()})
onnx_ms = (time.time() - start) * 1000 / 100

print("PyTorch:", round(torch_ms, 2), "ms per picture")
print("ONNX   :", round(onnx_ms, 2), "ms per picture")

PyTorch: 16.11 ms per picture
ONNX   : 2.31 ms per picture


## Summary

1. `torch.onnx.export` saves the model as one `.onnx` file.
2. ONNX Runtime runs it without PyTorch, with the same answers.
3. It is usually faster, so it is a good choice for a web app.

The smaller int8 file used by the Day 3 app is made by `scripts/train_and_export.py`
(quantization was explained in notebook 02).